# Gate on a fresh pipeline run (379 official photos, 6 queries, conf 0.22)
Data: `results/fresh_run/gate_rows_conf0.22.csv` from `Gate_Features_Run.ipynb`. Gate trained with folds that never contain the photo's group. Same photos as the dev set, so this tests the gate with new features, not new photos.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
here = pathlib.Path.cwd()
repo = next(p for p in [here, *here.parents] if (p / "src").is_dir() and (p / "train" / "checks").is_dir())
sys.path[:0] = [str(repo), str(repo / "src")]
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from train.checks.decision_gate import points_from_error, score_of_flags
d = pd.read_csv(repo / "tests/Test_Iteration_3/results/fresh_run/gate_rows_conf0.22.csv"); n = len(d)
P = points_from_error(d.error_sigma.values); y = (P > 0).astype(int)
for name, flags in [("current rule (bar 8)", d.inliers.values >= 8), ("always answer", np.ones(n, bool)), ("ceiling", P > 0)]:
    print("%-22s %.2f  wrong %d" % ((name,) + score_of_flags(P, flags)))

current rule (bar 8)   49.01  wrong 0
always answer          54.17  wrong 9
ceiling                57.02  wrong 0


In [2]:
d["li"] = np.log1p(d.inliers.clip(upper=200)); d["lr"] = np.log1p(d.runner_up_inliers)
d["ratio"] = d.inliers / (d.inliers + d.runner_up_inliers + 1)
sets = {"inliers+z": ["li", "z_chosen"], "+runner-up": ["li", "z_chosen", "lr", "ratio"], "+agreement": ["li", "z_chosen", "agree_queries"],
        "+peak sharpness": ["li", "z_chosen", "peak_sharpness", "peak_score"],
        "all": ["li", "z_chosen", "lr", "ratio", "agree_queries", "peak_sharpness", "peak_score", "height"]}
groups = {"height decile": pd.qcut(d.height, 10, labels=False, duplicates="drop").values, "photo-id block": d.photo.values // 40}
rows = []
for gname, g in groups.items():
    for sname, cols in sets.items():
        for mname, mk in [("logit", lambda: LogisticRegression(C=1, max_iter=3000)), ("gbm", lambda: GradientBoostingClassifier(n_estimators=60, max_depth=2, random_state=0))]:
            X = d[cols].values; X = (X - X.mean(0)) / X.std(0); flags = np.zeros(n, bool)
            for tr, te in GroupKFold(5).split(X, y, g):
                flags[te] = mk().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1] > 0.6
            s, w = score_of_flags(P, flags); rows.append([gname, sname, mname, round(s, 2), w])
res = pd.DataFrame(rows, columns=["folds by", "features", "model", "score/60", "wrong"])
res.pivot_table(index=["features", "model"], columns="folds by", values="score/60")

folds by               height decile  photo-id block
features        model                               
+agreement      gbm            56.39           54.81
                logit          56.23           56.07
+peak sharpness gbm            56.30           56.30
                logit          56.55           56.55
+runner-up      gbm            55.92           54.33
                logit          55.92           55.92
all             gbm            55.98           55.98
                logit          56.30           55.98
inliers+z       gbm            56.07           54.49
                logit          56.23           56.07

## Reading
- On these photos always answering already beats the bar-8 rule by about 5 marks; the bar throws away right answers (inliers 5 to 7 are mostly right here).
- A logistic gate on inliers + coarse z adds about 2 more marks over always-answering (56.1 to 56.5 against a ceiling of 57.0) with 1 to 2 wrong answers; peak sharpness helps slightly, runner-up and agreement do not. Boosted trees are less stable across photo-id folds.
- Same dev photos as before: confirm on photos from a different flight set or the Cartosat tests before trusting the exact numbers.